## 🔗 延伸学习

### 相关资源
- [Scikit-learn SVR 官方文档](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html)
- [SVM 理论详解](https://en.wikipedia.org/wiki/Support_vector_machine)

### 后续可以尝试的方向
1. **多元回归**：处理多个输入特征
2. **分类任务**：使用SVC处理分类问题
3. **集成方法**：结合SVM与其他算法（如随机森林）
4. **核函数自定义**：设计特定领域的核函数
5. **概率估计**：使用SVR+概率校准进行不确定性估计

---

**编写日期**: 2026年1月27日  
**难度等级**: 初学者  
**完成时间**: 约20-30分钟

## 📝 核心要点总结

### SVM回归的优势
- ✅ 对非线性问题处理能力强
- ✅ 通过核函数灵活处理复杂数据
- ✅ 泛化能力强，不容易过度拟合
- ✅ 在高维空间表现良好

### SVM回归的劣势
- ❌ 对大数据集处理速度较慢
- ❌ 超参数需要仔细调整
- ❌ 模型不易解释（黑盒模型）

### 实践建议
1. **数据预处理**：特征标准化/归一化是必须的
2. **选择核函数**：从RBF开始（默认），再尝试其他
3. **超参数调整**：使用GridSearchCV或RandomizedSearchCV
4. **验证方法**：使用交叉验证评估模型稳定性
5. **处理类别不平衡**：可以调整class_weight参数

### 何时使用SVM
- 特征维度较高（高于特征数）
- 样本数量适中（不超过几万）
- 需要非线性决策边界
- 对模型精度要求高

In [ ]:
# 生成密集的预测点以绘制平滑曲线
X_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
X_plot_scaled = scaler.transform(X_plot)

# 用原始模型和优化后的模型进行预测
y_plot_original = svm_model.predict(X_plot_scaled)
y_plot_optimized = best_model.predict(X_plot_scaled)
y_plot_true = np.sin(X_plot).ravel()

# 创建可视化图表
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 原始模型的回归曲线
ax1 = axes[0, 0]
ax1.scatter(X_train, y_train, color='blue', alpha=0.6, label='训练集', s=50)
ax1.scatter(X_test, y_test, color='red', alpha=0.6, label='测试集', s=50)
ax1.plot(X_plot, y_plot_original, 'g-', linewidth=2, label='SVM预测曲线')
ax1.plot(X_plot, y_plot_true, 'k--', linewidth=1.5, alpha=0.5, label='真实函数')
ax1.set_xlabel('X 特征')
ax1.set_ylabel('y 目标值')
ax1.set_title('原始SVM模型 (C=100, gamma=scale)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 优化后模型的回归曲线
ax2 = axes[0, 1]
ax2.scatter(X_train, y_train, color='blue', alpha=0.6, label='训练集', s=50)
ax2.scatter(X_test, y_test, color='red', alpha=0.6, label='测试集', s=50)
ax2.plot(X_plot, y_plot_optimized, 'purple', linewidth=2, label='SVM预测曲线（优化后）')
ax2.plot(X_plot, y_plot_true, 'k--', linewidth=1.5, alpha=0.5, label='真实函数')
ax2.set_xlabel('X 特征')
ax2.set_ylabel('y 目标值')
ax2.set_title(f"优化后SVM模型 (C={grid_search.best_params_['C']}, gamma={grid_search.best_params_['gamma']})")
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 测试集预测误差分析
ax3 = axes[1, 0]
errors_optimized = y_test - y_test_pred_best
ax3.scatter(y_test_pred_best, errors_optimized, alpha=0.6, s=80, color='purple')
ax3.axhline(y=0, color='k', linestyle='-', linewidth=1)
ax3.set_xlabel('预测值')
ax3.set_ylabel('残差 (实际 - 预测)')
ax3.set_title('残差图 (优化后模型)')
ax3.grid(True, alpha=0.3)

# 4. 预测值 vs 实际值
ax4 = axes[1, 1]
ax4.scatter(y_test, y_test_pred_best, alpha=0.6, s=80, color='orange')
min_val = min(y_test.min(), y_test_pred_best.min())
max_val = max(y_test.max(), y_test_pred_best.max())
ax4.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='完美预测线')
ax4.set_xlabel('实际值')
ax4.set_ylabel('预测值')
ax4.set_title('预测值 vs 实际值 (优化后模型)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 可视化完成")

## 8️⃣ 回归曲线可视化

通过可视化来展示SVM回归模型的拟合效果。

In [ ]:
# 用最优模型进行预测
y_train_pred_best = best_model.predict(X_train_scaled)
y_test_pred_best = best_model.predict(X_test_scaled)

# 评估优化后的模型
print("\n优化后模型的性能:")
train_metrics_best = evaluate_model(y_train, y_train_pred_best, "训练")
test_metrics_best = evaluate_model(y_test, y_test_pred_best, "测试")

# 性能对比
print("\n" + "="*60)
print("原始模型 vs 优化后模型 对比:")
print("="*60)
print(f"{'指标':<15} {'原始模型':<15} {'优化后':<15} {'提升':<15}")
print("-"*60)
improvement = (test_metrics_best['R2'] - test_metrics['R2']) / abs(test_metrics['R2']) * 100 if test_metrics['R2'] != 0 else 0
print(f"{'测试R²':<15} {test_metrics['R2']:<15.6f} {test_metrics_best['R2']:<15.6f} {improvement:>+.2f}%")

improvement_rmse = (test_metrics['RMSE'] - test_metrics_best['RMSE']) / test_metrics['RMSE'] * 100
print(f"{'测试RMSE':<15} {test_metrics['RMSE']:<15.6f} {test_metrics_best['RMSE']:<15.6f} {improvement_rmse:>+.2f}%")

In [ ]:
# 定义要搜索的超参数网格
param_grid = {
    'C': [0.1, 1, 10, 100, 1000],           # 正则化参数
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],  # 核函数参数
    'epsilon': [0.01, 0.05, 0.1, 0.2]      # 误差容限
}

# 创建网格搜索对象（使用5折交叉验证）
grid_search = GridSearchCV(
    SVR(kernel='rbf'),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,  # 使用所有CPU核心
    verbose=1
)

# 执行网格搜索
print("正在进行超参数网格搜索...（这可能需要几秒钟）\n")
grid_search.fit(X_train_scaled, y_train)

print("\n" + "="*60)
print("网格搜索完成！")
print("="*60)
print(f"\n最优参数组合: {grid_search.best_params_}")
print(f"最优模型的交叉验证R²分数: {grid_search.best_score_:.6f}")

# 获取最优模型
best_model = grid_search.best_estimator_

## 7️⃣ 超参数优化

使用网格搜索(GridSearchCV)找到最优的超参数组合。这个过程会尝试不同的参数组合并选择性能最好的。

In [ ]:
# 计算评估指标
def evaluate_model(y_true, y_pred, set_name):
    """计算并打印模型评估指标"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{set_name}集评估结果:")
    print(f"  MSE (均方误差):  {mse:.6f}")
    print(f"  RMSE (均方根误差): {rmse:.6f}")
    print(f"  MAE (平均绝对误差): {mae:.6f}")
    print(f"  R² (决定系数):   {r2:.6f}")
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

# 评估训练集和测试集
train_metrics = evaluate_model(y_train, y_train_pred, "训练")
test_metrics = evaluate_model(y_test, y_test_pred, "测试")

print("\n" + "="*50)
print("模型性能分析:")
if test_metrics['R2'] > 0.8:
    print("✓ 模型性能优秀！R²值很高，模型泛化能力强")
elif test_metrics['R2'] > 0.5:
    print("✓ 模型性能良好，可进一步优化")
else:
    print("⚠ 模型性能一般，建议调整超参数")

## 6️⃣ 评估模型性能

计算多个评估指标来了解模型的预测能力。

### 评估指标说明
- **MSE (均方误差)**：$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$
- **RMSE (均方根误差)**：$\text{RMSE} = \sqrt{\text{MSE}}$，与y的单位相同
- **MAE (平均绝对误差)**：$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$，对异常值不敏感
- **R² (决定系数)**：$R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$，范围0-1，越接近1越好

In [ ]:
# 使用训练好的模型进行预测
y_train_pred = svm_model.predict(X_train_scaled)
y_test_pred = svm_model.predict(X_test_scaled)

print("✓ 预测完成")
print("\n前5个测试样本的预测结果对比:")
print(f"{'实际值':>10} | {'预测值':>10} | {'误差':>10}")
print("-" * 35)
for i in range(min(5, len(y_test))):
    error = y_test[i] - y_test_pred[i]
    print(f"{y_test[i]:>10.4f} | {y_test_pred[i]:>10.4f} | {error:>10.4f}")

## 5️⃣ 模型预测

使用训练好的模型对测试数据进行预测。

In [ ]:
# 创建SVM回归模型（使用RBF核）
# 参数说明：
# - kernel='rbf': 使用高斯径向基函数，适合非线性问题
# - C=100: 正则化参数，较大的C意味着较小的容限
# - gamma='scale': 自动计算gamma为1/(n_features*X.var())
# - epsilon=0.1: 误差容限，在此范围内的误差被忽略

svm_model = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)

# 用训练数据训练模型
svm_model.fit(X_train_scaled, y_train)

print("✓ SVM模型已训练")
print(f"模型使用的支持向量数: {len(svm_model.support_vectors_)}")
print(f"训练集中的总样本数: {len(X_train)}")
print(f"支持向量比例: {len(svm_model.support_vectors_)/len(X_train)*100:.1f}%")

## 4️⃣ 构建SVM回归模型

### 什么是SVM？
支持向量机(Support Vector Machine, SVM)是一种强大的监督学习算法。在回归任务中，SVR(Support Vector Regression)试图找到一个超平面，使得尽可能多的数据点落在由ε(epsilon)定义的间隔带内。

### 核函数(Kernel)类型
- **Linear (线性)**：适合线性可分的数据
- **RBF (径向基函数)**：适合非线性数据，是默认选择
- **Polynomial (多项式)**：处理多项式关系的数据

### 关键超参数
- **C**：正则化参数，控制模型复杂度（C越大，模型越复杂）
- **gamma**：只对RBF、Poly核有效，定义一个训练样本的影响范围

In [ ]:
plt.figure(figsize=(12, 4))

# 原始数据分布
plt.subplot(1, 2, 1)
plt.scatter(X_train, y_train, color='blue', alpha=0.6, label='训练集', s=50)
plt.scatter(X_test, y_test, color='red', alpha=0.6, label='测试集', s=50)
plt.xlabel('X 特征')
plt.ylabel('y 目标值')
plt.title('原始数据分布')
plt.legend()
plt.grid(True, alpha=0.3)

# 真实函数曲线
plt.subplot(1, 2, 2)
X_true = np.linspace(0, 10, 200).reshape(-1, 1)
y_true = np.sin(X_true).ravel()
plt.plot(X_true, y_true, 'g-', linewidth=2, label='真实函数: y=sin(x)')
plt.scatter(X_train, y_train, color='blue', alpha=0.6, label='训练数据', s=50)
plt.xlabel('X 特征')
plt.ylabel('y 目标值')
plt.title('数据与真实函数对比')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 数据已可视化")

## 3️⃣ 数据可视化

让我们先观察原始数据的分布，了解我们要拟合的函数形状。

In [ ]:
# 特征标准化（使用训练集的统计量）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("特征标准化完成")
print(f"训练集特征均值: {X_train_scaled.mean():.4f}")
print(f"训练集特征标准差: {X_train_scaled.std():.4f}")

### 特征标准化

**为什么需要标准化？**
- SVM对特征尺度敏感，标准化可以改善模型性能
- 将特征转换为均值为0、标准差为1的分布

In [3]:
# 设置随机种子以确保结果可复现
np.random.seed(42)

# 生成示例数据：y = sin(x) + noise
X = np.linspace(0, 10, 100).reshape(-1, 1)  # 100个样本，1个特征
y = np.sin(X).ravel() + np.random.normal(0, 0.1, X.shape[0])  # 加入噪声

# 分割训练集和测试集 (80% 训练, 20% 测试)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"训练集大小: {X_train.shape[0]} 样本")
print(f"测试集大小: {X_test.shape[0]} 样本")
print(f"特征数: {X_train.shape[1]}")

训练集大小: 80 样本
测试集大小: 20 样本
特征数: 1


## 2️⃣ 生成和准备数据

我们将生成一个示例数据集来演示SVM回归的过程。这个数据集包含非线性关系，更好地展示SVM的优势。

In [2]:
# 导入必要的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体（Windows）
plt.rcParams['font.sans-serif'] = ['SimHei']  # 黑体
plt.rcParams['axes.unicode_minus'] = False     # 正常显示负号

print("✓ 所有库导入成功！")

✓ 所有库导入成功！


## 1️⃣ 导入必要的库

首先，我们需要导入Python中用于数据处理、可视化和机器学习的必要库。

# 支持向量机(SVM)回归预测初学者教程

本教程将引导你学习如何使用Python和Scikit-learn实现支持向量机回归预测。

## 学习目标
- 理解SVM回归的基本原理
- 掌握数据预处理和分割方法
- 学会构建和训练SVM回归模型
- 能够评估和优化模型性能
- 掌握超参数调整和可视化技巧

---